# 🎨 Probador Interactivo: Stable Diffusion v1.4 vs. LoRA (Old Book Illustrations)
### **Máster en Inteligencia Artificial e IA Generativa — PontIA**
- **Alumno:** Daniel Aguilera González ([@Twinber](https://huggingface.co/Twinber))
- **Modelo Base:** `CompVis/stable-diffusion-v1-4`
- **Adaptador LoRA:** [`Twinber/sd-v1-4-oldbook-illustrations-lora`](https://huggingface.co/Twinber/sd-v1-4-oldbook-illustrations-lora)

---
### 📌 Propósito de este Cuaderno
Este cuaderno está diseñado para **experimentar de forma ágil y visual en Google Colab con aceleración GPU (T4/L4/A100)**.
Permite comparar en paralelo y con la misma semilla aleatoria el comportamiento del **modelo base puro** frente al **modelo fine-tuneado con LoRA**, observando cómo se transfiere el estilo de grabado histórico, achurado (*cross-hatching*) y textura litográfica del siglo XIX.

## 1. Instalación de dependencias
Instalamos `diffusers`, `transformers`, `accelerate` y `safetensors` para manejar el pipeline y la inyección dinámica de LoRA.

In [ ]:
# 1. Desinstalamos torchao para eliminar el conflicto de importación en Colab
!pip uninstall -y -q torchao

# 2. Instalación de librerías esenciales (incluyendo peft para gestión de adaptadores LoRA)
!pip install -q diffusers transformers accelerate peft safetensors


## 2. Comprobación de Entorno y GPU

In [ ]:
import os
import time
import torch
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

# 🛡️ Blindaje definitivo contra el conflicto de torchao en Google Colab
# (Evita el fallo de FqnToConfig sin necesidad de reiniciar la sesión de Colab)
import diffusers.utils.import_utils
diffusers.utils.import_utils._torchao_available = False
diffusers.utils.import_utils.is_torchao_available = lambda: False

from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

# Comprobamos acelerador
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Dispositivo activo: {device.upper()}")
if device == "cuda":
    print(f"   GPU detectada: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ No se ha detectado GPU. Ve a Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU.")


## 3. Carga del Modelo Base y del Adaptador LoRA
Cargamos `CompVis/stable-diffusion-v1-4` en precisión `float16` y le incorporamos el adaptador LoRA publicado en Hugging Face Hub.

> **💡 Eficiencia de arquitectura:** Usamos el sistema de adaptadores de `diffusers`. No duplicamos la UNet en memoria: con `pipe.set_adapters(..., adapter_weights=[0.0])` evaluamos el modelo base puro, y con `adapter_weights=[1.0]` evaluamos el estilo de grabado completo.

In [ ]:
model_id = "CompVis/stable-diffusion-v1-4"
lora_repo_id = "Twinber/sd-v1-4-oldbook-illustrations-lora"

print("⏳ Cargando modelo base Stable Diffusion v1.4...")
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,
    requires_safety_checker=False
).to(device)

# Configuramos DPMSolverMultistepScheduler para inferencia ultra-rápida (15-20 pasos en ~2 segundos en GPU)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

print("⏳ Cargando adaptador LoRA desde Hugging Face Hub...")
pipe.load_lora_weights(lora_repo_id, adapter_name="oldbook_illustrations")
print("✅ ¡Pipeline cargado y listo en memoria!")

## 4. Función de Generación y Comparativa Visual (Antes vs Después)

In [ ]:
def generar_comparativa(
    prompt,
    negative_prompt="blurry, low quality, distorted, deformed, text, watermark, photo, photorealistic, 3d render",
    seed=42,
    num_inference_steps=20,
    guidance_scale=7.5,
    lora_weight=1.0,
    guardar=True
):
    """
    Genera dos imágenes con la misma semilla y prompt:
    1. Modelo Base (LoRA desactivado: weight = 0.0)
    2. Modelo con LoRA (weight = lora_weight)
    Y muestra un panel comparativo lado a lado.
    """
    print(f"\n🎨 Prompt: '{prompt}'")
    print(f"   Semilla: {seed} | Pasos: {num_inference_steps} | Peso LoRA: {lora_weight}")
    
    # 1. Generar ANTES (Base)
    pipe.set_adapters(["oldbook_illustrations"], adapter_weights=[0.0])
    gen_base = torch.Generator(device=device).manual_seed(seed)
    t0 = time.time()
    img_base = pipe(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=gen_base
    ).images[0]
    t_base = time.time() - t0
    
    # 2. Generar DESPUÉS (LoRA)
    pipe.set_adapters(["oldbook_illustrations"], adapter_weights=[float(lora_weight)])
    gen_lora = torch.Generator(device=device).manual_seed(seed)
    t1 = time.time()
    img_lora = pipe(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=gen_lora
    ).images[0]
    t_lora = time.time() - t1
    
    print(f"⏱️ Tiempos: Base: {t_base:.2f}s | LoRA: {t_lora:.2f}s")
    
    # Montaje comparativo con cartelas
    w, h = img_base.size
    header_h = 50
    panel = Image.new("RGB", (w * 2, h + header_h), (245, 245, 245))
    panel.paste(img_base, (0, header_h))
    panel.paste(img_lora, (w, header_h))
    
    draw = ImageDraw.Draw(panel)
    draw.line([(w, 0), (w, h + header_h)], fill=(160, 160, 160), width=2)
    draw.line([(0, header_h), (w * 2, header_h)], fill=(160, 160, 160), width=1)
    draw.text((25, 16), f"ANTES: SD v1.4 Base (Sin LoRA) — {t_base:.1f}s", fill=(40, 40, 40))
    draw.text((w + 25, 16), f"DESPUES: SD v1.4 + LoRA OldBook (Peso={lora_weight}) — {t_lora:.1f}s", fill=(10, 80, 160))
    
    if guardar:
        sanitized = "".join(c if c.isalnum() else "_" for c in prompt[:25])
        filename = f"comparativa_{sanitized}_seed{seed}.png"
        panel.save(filename)
        print(f"💾 Panel guardado en disco como: {filename}")
        
    # Visualización
    plt.figure(figsize=(15, 8))
    plt.imshow(panel)
    plt.axis("off")
    plt.title(f"Prompt: {prompt}\nSeed: {seed} | Steps: {num_inference_steps}", fontsize=11, pad=12)
    plt.show()
    
    return img_base, img_lora, panel

## 5. Pruebas Manuales: Escribe y Ejecuta tus Prompts
Modifica las variables de la siguiente celda tantas veces como quieras para probar cualquier idea:

In [ ]:
# ==============================================================================
# ✍️ MODIFICA ESTOS PARÁMETROS PARA TUS PRUEBAS:
# ==============================================================================

# Ideas de prompts sugeridos:
# 1. "a man walking his dog by the shores of a lake, tranquil landscape"
# 2. "a medieval gothic castle on a misty cliff, antique vintage engraving"
# 3. "a 19th century tall sailing ship in a rough stormy ocean"
# 4. "a vintage steam locomotive train crossing an arched bridge in the mountains"
# 5. "an antique botanical illustration of an exotic carnivorous plant, detailed"
# 6. "portrait of an old astronomer looking through a brass telescope, night sky"

PROMPT = "a man walking his dog by the shores of a lake, tranquil landscape"
SEMILLA = 42
PASOS = 20
PESO_LORA = 1.0   # 1.0 = estilo de grabado completo; 0.5 = mezcla suave; 1.3 = grabado muy marcado

img_base, img_lora, panel = generar_comparativa(
    prompt=PROMPT,
    seed=SEMILLA,
    num_inference_steps=PASOS,
    lora_weight=PESO_LORA
);

## 6. Panel Interactivo con Widgets (Opcional)
Si prefieres una interfaz interactiva con caja de texto, deslizadores de pasos y selector de semillas dentro del propio Colab, ejecuta esta celda:

In [ ]:
import ipywidgets as widgets
from IPython.display import display

txt_prompt = widgets.Textarea(
    value="a medieval gothic castle on a misty cliff, antique vintage engraving",
    description="Prompt:",
    layout=widgets.Layout(width="80%", height="70px")
)

num_seed = widgets.IntText(value=42, description="Semilla:")
slider_steps = widgets.IntSlider(value=20, min=10, max=35, step=1, description="Pasos:")
slider_weight = widgets.FloatSlider(value=1.0, min=0.0, max=1.5, step=0.1, description="Peso LoRA:")
btn_run = widgets.Button(description="🚀 Generar Comparativa", button_style="primary", icon="play")
output_area = widgets.Output()

def ejecutar_click(b):
    with output_area:
        output_area.clear_output()
        generar_comparativa(
            prompt=txt_prompt.value,
            seed=num_seed.value,
            num_inference_steps=slider_steps.value,
            lora_weight=slider_weight.value
        )

btn_run.on_click(ejecutar_click)

display(
    widgets.VBox([
        txt_prompt,
        widgets.HBox([num_seed, slider_steps, slider_weight]),
        btn_run,
        output_area
    ])
)

## 7. Galería de Demostración Automática (Varios Estilos)
Ejecuta esta celda si deseas generar automáticamente una galería de 3 conceptos distintos bajo el estilo de grabado:

In [ ]:
prompts_galeria = [
    ("a 19th century tall sailing ship in a rough stormy ocean", 101),
    ("a vintage steam locomotive train crossing an arched bridge in the mountains", 202),
    ("an antique botanical illustration of an exotic orchid flower with leaves", 303),
]

for p, s in prompts_galeria:
    generar_comparativa(prompt=p, seed=s, num_inference_steps=20, lora_weight=1.0)